# 아산시 돌봄 데이터 전처리 파이프라인

| # | 파일/폴더 | 형식 | 인코딩 | 비고 |
|---|---|---|---|---|
| 1 | 읍면동별 인구총조사 데이터 | CSV | utf-8 | 2행 멀티헤더 (연도/항목) |
| 2 | 읍면동별_연령별_성별_주민등록_인구 | CSV | cp949 | sparse 연도 (2015,2020,2023,2024) |
| 3 | 장래인구추계_충청남도_2014_2052 | CSV | utf-8 | 4개 시나리오, 중위만 사용 |
| 4 | 성별연령별 장기요양 등급 판정인정 현황 | CSV | utf-8 | 3행 멀티헤더 (연도/급여유형/등급) |
| 5 | 충청남도_재가노인복지시설 현황_20240630 | CSV | cp949 | 아산시 필터링 필요 |
| 6 | 장기요양시설별 현황 | ZIP→XLSX | - | 연도별 시설/입소/인력 |
| 7 | 아산시 기초생활보장 연령별 수급자수 | ZIP→CSV | cp949 | 132개 월별 파일 |
| 8 | 2024 노인복지시설 현황 | ZIP→XLSX | - | 시군구별 총괄표 |
| 9 | 전국 병의원 및 약국 | ZIP→XLSX | - | 시점별 7개 ZIP |
| 10 | 지역사회건강조사 | 폴더 | - | 아산시 건강지표 |
| 11 | 주민등록인구기타현황 | 폴더 | - | 세대수 등 |
| 12 | 집계구·행정구역별 통계 (인구, 가구) | ZIP | - | 소지역 통계 |

## 0. 환경 설정

In [52]:
import pandas as pd
import numpy as np
import glob
import zipfile
import os
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 30)
pd.set_option('display.width', 120)

# ── 경로 설정 ──
DATA_DIR = Path(r'C:\Users\HP\IdeaProjects\sundo\asan_care\asan_care\데이터')
OUT_DIR = DATA_DIR.parent / 'preprocessed'
OUT_DIR.mkdir(exist_ok=True)

print(f'데이터 경로: {DATA_DIR}')
print(f'출력 경로:   {OUT_DIR}')
print(f'파일 목록:')
for p in sorted(DATA_DIR.iterdir()):
    print(f'  {"📁" if p.is_dir() else "📄"} {p.name} ({p.stat().st_size/1024:.0f}KB)' if p.is_file() else f'  📁 {p.name}/')

데이터 경로: C:\Users\HP\IdeaProjects\sundo\asan_care\asan_care\데이터
출력 경로:   C:\Users\HP\IdeaProjects\sundo\asan_care\asan_care\preprocessed
파일 목록:
  📁 2024 노인복지시설 현황/
  📄 2024 노인복지시설 현황.zip (39066KB)
  📄 ~$아산시_돌봄_데이터_명세서.xlsx (0KB)
  📄 성별연령별 장기요양 등급 판정인정 현황.csv (13KB)
  📁 아산시 기초생활보장 연령별 수급자수(일반)(201401~202412)/
  📄 아산시 기초생활보장 연령별 수급자수(일반)(201401~202412).zip (12439KB)
  📄 아산시_돌봄_데이터_명세서.xlsx (36KB)
  📄 읍면동별 인구총조사 데이터.csv (4KB)
  📄 읍면동별_연령별_성별_주민등록_인구.csv (301KB)
  📁 장기요양시설별 현황/
  📄 장래인구추계_충청남도_2014_2052.csv (99KB)
  📁 전국 병의원 및 약국/
  📁 주민등록인구기타현황/
  📁 지역사회건강조사/
  📁 집계구·행정구역별 통계 (인구, 가구)/
  📄 집계구·행정구역별 통계 (인구, 가구).zip (2095KB)
  📄 충청남도_재가노인복지시설 현황_20240630.csv (75KB)


---
## 1. 읍면동별 인구총조사 (2015, 2020)

**문제점:** 2행 멀티헤더 (row0=연도, row1=항목명), `X`=비공개, `-`=해당없음

In [53]:
# ── 로딩 ──
raw_census = pd.read_csv(DATA_DIR / '읍면동별 인구총조사 데이터.csv', encoding='utf-8', header=None)
print(f'원본: {raw_census.shape}')
raw_census.head(3)

원본: (19, 41)


,0,1,2,3,4,5,6,7,8,9,...,31,32,33,34,35,36,37,38,39,40
0,행정구역별(읍면동),2015,2015,2015,2015,2015,2015,2015,2015,2015,...,2020,2020,2020,2020,2020,2020,2020,2020,2020,2020
1,행정구역별(읍면동),총인구 (명),남자 (명),여자 (명),내국인-계 (명),내국인-남자 (명),내국인-여자 (명),외국인-계 (명),외국인-남자 (명),외국인-여자 (명),...,일반가구 (가구),집단가구 (가구),외국인가구 (가구),주택_계 (호),주택_단독주택 (호),주택_아파트 (호),주택_연립주택 (호),주택_다세대주택 (호),주택_비주거용 건물 내 주택 (호),주택 이외의 거처_계 (호)
2,염치읍,7643,4052,3591,7287,3792,3495,356,260,96,...,2723,X,63,3017,1491,1283,51,163,29,159


In [54]:
# ── 멀티헤더 파싱 → long format ──
years = raw_census.iloc[0, 1:].values
metrics = raw_census.iloc[1, 1:].values

data = raw_census.iloc[2:].copy().reset_index(drop=True)
data.columns = ['읍면동'] + [f'{str(y).split(".")[0]}_{m}' for y, m in zip(years, metrics)]

records = []
for _, row in data.iterrows():
    dong = row['읍면동']
    for col in data.columns[1:]:
        year, metric = col.split('_', 1)
        val = row[col]
        if val in ('X', '-', 'x', '…'):
            val = np.nan
        else:
            try:
                val = float(str(val).replace(',', ''))
            except ValueError:
                val = np.nan
        records.append({'읍면동': dong, '연도': int(year), '항목': metric, '값': val})

df_census = pd.DataFrame(records)
print(f'정제 완료: {df_census.shape[0]:,}행 | 읍면동 {df_census["읍면동"].nunique()}개 | 연도 {sorted(df_census["연도"].unique())}')
df_census.head()

정제 완료: 680행 | 읍면동 17개 | 연도 [np.int64(2015), np.int64(2020)]


,읍면동,연도,항목,값
0,염치읍,2015,총인구 (명),7643.0
1,염치읍,2015,남자 (명),4052.0
2,염치읍,2015,여자 (명),3591.0
3,염치읍,2015,내국인-계 (명),7287.0
4,염치읍,2015,내국인-남자 (명),3792.0


---
## 2. 읍면동별 연령별 성별 주민등록 인구

**문제점:** cp949 인코딩, 연도 sparse(2016~2019 결측), 컬럼명에 '년' 포함, 소계 라벨(합계, 65세이상, 85세이상 등) 혼재

In [55]:
# ── 로딩 & 컬럼명 정리 ──
df_resident = pd.read_csv(DATA_DIR / '읍면동별_연령별_성별_주민등록_인구.csv', encoding='cp949')

# Unnamed 컬럼 및 단위 제거
df_resident = df_resident.drop(columns=[c for c in df_resident.columns if 'Unnamed' in str(c)], errors='ignore')
df_resident = df_resident.drop(columns=['단위'], errors='ignore')

# '2015 년' → 2015
year_cols = [c for c in df_resident.columns if '년' in str(c)]
rename_map = {c: int(c.replace('년', '').strip()) for c in year_cols}
df_resident = df_resident.rename(columns=rename_map)

print(f'컬럼: {list(df_resident.columns)}')
print(f'연령별 라벨: {df_resident["연령별"].unique()}')
df_resident.head()

컬럼: ['행정구역별(읍면동)', '연령별', '항목', 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
연령별 라벨: <StringArray>
[    '합계',   '0~4세',   '5~9세', '10~14세', '15~19세', '20~24세', '25~29세', '30~34세', '35~39세', '40~44세', '45~49세',
 '50~54세', '55~59세', '60~64세', '65~69세', '70~74세', '75~79세', '80~84세', '85~89세', '90~94세', '95~99세', '100세이상',
  '15세미만', '15~64세',  '65세이상',  '85세이상',   '평균연령',   '중위연령']
Length: 28, dtype: str


,행정구역별(읍면동),연령별,항목,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,염치읍,합계,총인구(명),7643,NaN,NaN,NaN,NaN,6521,NaN,NaN,6094,5851
1,염치읍,합계,총인구_남자(명),4052,NaN,NaN,NaN,NaN,3455,NaN,NaN,3261,3165
2,염치읍,합계,총인구_여자(명),3591,NaN,NaN,NaN,NaN,3066,NaN,NaN,2833,2686
3,염치읍,합계,총인구_성비,112.8,NaN,NaN,NaN,NaN,112.7,NaN,NaN,115.1,117.8
4,염치읍,합계,내국인(명),7287,NaN,NaN,NaN,NaN,6209,NaN,NaN,5737,5484


In [56]:
# ── wide → long ──
id_cols = ['행정구역별(읍면동)', '연령별', '항목']
val_cols = sorted([c for c in df_resident.columns if isinstance(c, int)])

df_resident_long = df_resident.melt(id_vars=id_cols, value_vars=val_cols, var_name='연도', value_name='값')

# 값 정제
df_resident_long['값'] = pd.to_numeric(
    df_resident_long['값'].astype(str).str.replace(',', '').replace({'X': np.nan, '-': np.nan, '…': np.nan}),
    errors='coerce'
)
df_resident_long = df_resident_long.dropna(subset=['값'])
df_resident_long = df_resident_long.rename(columns={'행정구역별(읍면동)': '읍면동'})

print(f'정제 완료: {df_resident_long.shape[0]:,}행 | 읍면동 {df_resident_long["읍면동"].nunique()}개 | 연도 {sorted(df_resident_long["연도"].unique())}')
df_resident_long.head()

정제 완료: 14,382행 | 읍면동 17개 | 연도 [2015, 2020, 2023, 2024]


,읍면동,연령별,항목,연도,값
0,염치읍,합계,총인구(명),2015,7643.0
1,염치읍,합계,총인구_남자(명),2015,4052.0
2,염치읍,합계,총인구_여자(명),2015,3591.0
3,염치읍,합계,총인구_성비,2015,112.8
4,염치읍,합계,내국인(명),2015,7287.0


---
## 3. 장래인구추계 충청남도 (2014~2052)

**문제점:** 4개 시나리오 혼재 (모두 '중위' 포함 → `startswith`로 필터), `80세이상` 소계와 세부 연령 중복

In [57]:
# ── 로딩 & 중위 추계 필터 ──
df_proj_raw = pd.read_csv(DATA_DIR / '장래인구추계_충청남도_2014_2052.csv', encoding='utf-8')

# ⚠️ 주의: '고위 추계(...국내이동-중위)' 등에도 '중위'가 포함됨 → startswith 필수
df_proj = df_proj_raw[df_proj_raw['시나리오별(1)'].str.startswith('중위', na=False)].copy()
print(f'전체 {len(df_proj_raw)}행 → 중위만 {len(df_proj)}행')
print(f'연령대: {df_proj["연령별(1)"].unique()}')

전체 276행 → 중위만 69행
연령대: <StringArray>
[       '계',   '0 - 4세',   '5 - 9세', '10 - 14세', '15 - 19세', '20 - 24세', '25 - 29세', '30 - 34세', '35 - 39세',
 '40 - 44세', '45 - 49세', '50 - 54세', '55 - 59세', '60 - 64세', '65 - 69세', '70 - 74세', '75 - 79세',    '80세이상',
 '80 - 84세', '85 - 89세', '90 - 94세', '95 - 99세',  '100세 이상']
Length: 23, dtype: str


In [58]:
# ── wide → long ──
year_cols = [c for c in df_proj.columns if c.isdigit()]
id_cols = ['시나리오별(1)', '시도별(1)', '성별(1)', '연령별(1)']

df_proj_long = df_proj.melt(id_vars=id_cols, value_vars=year_cols, var_name='연도', value_name='인구수')
df_proj_long['연도'] = df_proj_long['연도'].astype(int)
df_proj_long['인구수'] = pd.to_numeric(
    df_proj_long['인구수'].astype(str).str.replace(',', ''), errors='coerce'
)

df_proj_long = df_proj_long.rename(columns={
    '시도별(1)': '시도', '성별(1)': '성별', '연령별(1)': '연령대'
}).drop(columns=['시나리오별(1)'])

print(f'정제 완료: {df_proj_long.shape[0]:,}행 | 연도 {df_proj_long["연도"].min()}~{df_proj_long["연도"].max()}')
df_proj_long.head()

정제 완료: 2,691행 | 연도 2014~2052


,시도,성별,연령대,연도,인구수
0,충청남도,계,계,2014,2087600
1,충청남도,계,0 - 4세,2014,98885
2,충청남도,계,5 - 9세,2014,96922
3,충청남도,계,10 - 14세,2014,109870
4,충청남도,계,15 - 19세,2014,138950


---
## 4. 성별연령별 장기요양 등급 판정/인정 현황

**문제점:** 3행 멀티헤더(연도/급여유형/등급), 연도 ffill 필요, 420개 데이터 컬럼 → 연도×5카테고리×7등급

In [59]:
# ── 3행 멀티헤더 파싱 ──
raw_ltc = pd.read_csv(
    DATA_DIR / '성별연령별 장기요양 등급 판정인정 현황.csv',
    encoding='utf-8', header=None
)

row_year = raw_ltc.iloc[0, 3:].values
row_category = raw_ltc.iloc[1, 3:].values   # 계/일반/감경/의료급여/기초
row_grade = raw_ltc.iloc[2, 3:].values       # 계/1등급/2등급/.../등급외

# 연도 forward fill
years_filled = pd.Series(row_year).replace('', np.nan).ffill().values

data = raw_ltc.iloc[3:].copy().reset_index(drop=True)
meta_cols = ['시도', '시군구', '성별']
data.columns = meta_cols + list(range(len(row_year)))

print(f'헤더 구조: {len(set(years_filled))}개 연도 × {len(set(row_category))}개 카테고리 × {len(set(row_grade))}개 등급')
print(f'데이터 행: {len(data)}행')

헤더 구조: 11개 연도 × 5개 카테고리 × 8개 등급
데이터 행: 3행


In [60]:
# ── long format 변환 ──
records = []
for _, row in data.iterrows():
    for i in range(len(row_year)):
        val = row[i]
        if pd.isna(val) or str(val).strip() in ('', '-', 'X'):
            continue
        try:
            val = int(float(str(val).replace(',', '')))
        except (ValueError, TypeError):
            continue
        records.append({
            '시도': row['시도'], '시군구': row['시군구'], '성별': row['성별'],
            '연도': int(float(str(years_filled[i]))),
            '급여유형': str(row_category[i]),
            '등급': str(row_grade[i]),
            '인원': val,
        })

df_ltc = pd.DataFrame(records)
print(f'정제 완료: {df_ltc.shape[0]:,}행 | 연도 {sorted(df_ltc["연도"].unique())}')
print(f'등급: {df_ltc["등급"].unique()}')
df_ltc.head()

정제 완료: 1,253행 | 연도 [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
등급: <StringArray>
['계', '1등급', '2등급', '3등급', '4등급', '5등급', '등급외', '인지지원등급']
Length: 8, dtype: str


,시도,시군구,성별,연도,급여유형,등급,인원
0,충남,아산시,합계,2014,계,계,2424
1,충남,아산시,합계,2014,계,1등급,135
2,충남,아산시,합계,2014,계,2등급,387
3,충남,아산시,합계,2014,계,3등급,689
4,충남,아산시,합계,2014,계,4등급,510


---
## 5. 충청남도 재가노인복지시설 현황

**문제점:** cp949 인코딩, 충남 전체 데이터 → 아산시 필터링, 주소에서 읍면동 파싱 필요

In [61]:
# ── 로딩 & 아산시 필터 ──
df_facility_all = pd.read_csv(
    DATA_DIR / '충청남도_재가노인복지시설 현황_20240630.csv', encoding='cp949'
)
df_facility = df_facility_all[df_facility_all['시군'].str.contains('아산', na=False)].copy()

# 주소에서 읍면동 추출
def extract_dong(addr):
    if pd.isna(addr):
        return None
    for p in str(addr).split():
        for suffix in ('읍', '면', '동', '리'):
            if p.endswith(suffix) and len(p) > 1:
                return p
    return None

df_facility['읍면동'] = df_facility['주소'].apply(extract_dong)

print(f'충남 전체: {len(df_facility_all)}개 → 아산시: {len(df_facility)}개')
print(f'유형별:')
print(df_facility['종류'].value_counts().to_string())
df_facility.head()

충남 전체: 780개 → 아산시: 84개
유형별:
종류
방문요양         36
방문목욕         20
주야간보호        20
재가노인지원서비스     4
방문간호          3
복지용구          1


,시군,종류,시설명,주소,전화번호,읍면동
246,아산시,재가노인지원서비스,온양노인복지센터,충청남도 아산시 모종로 110 (모종동),041-543-4411,NaN
247,아산시,방문목욕,아산노인복지센터,충청남도 아산시 도고면 도고면로147번길 15 (도고면),041-544-7775,도고면
248,아산시,방문요양,아산노인복지센터,충청남도 아산시 도고면 도고면로147번길 15 (도고면),041-544-7775,도고면
249,아산시,방문간호,나눔케어,충청남도 아산시 청운로84번길 15 2층 (온천동),041-544-4328,NaN
250,아산시,방문목욕,나눔케어,충청남도 아산시 청운로84번길 15 2층 (온천동),041-544-4328,NaN


---
## 6. 장기요양시설별 현황 (ZIP → XLSX)

연도별 장기요양기관 일반현황/입소인원/인력현황. 아산시(시군구코드) 필터링 필요.

In [62]:
# ── ZIP 내 XLSX 파일 목록 확인 ──
ltc_facility_dir = DATA_DIR / '장기요양시설별 현황'

xlsx_files = sorted(ltc_facility_dir.glob('*.xlsx'))
print(f'XLSX 파일 {len(xlsx_files)}개:')
for f in xlsx_files:
    print(f'  {f.name} ({f.stat().st_size/1024:.0f}KB)')

XLSX 파일 5개:
  국민건강보험공단_장기요양기관 시설별 현황(2015년).xlsx (1525KB)
  국민건강보험공단_장기요양기관 시설별 현황(2020년).xlsx (5817KB)
  국민건강보험공단_장기요양기관 시설별 현황.xlsx (5717KB)
  국민건강보험공단_장기요양기관 시설별 현황_20240716.xlsx (6677KB)
  국민건강보험공단_장기요양기관 시설별 현황_20250401.xlsx (6437KB)


In [63]:
# ── 최신 파일 로딩 & 아산시 필터 ──
# 시트 구조: 일반현황, 입소인원, 인력현황, 기관유형코드 정의
latest_file = xlsx_files[-1]  # 가장 최신
print(f'로딩: {latest_file.name}')

xls = pd.ExcelFile(latest_file)
print(f'시트: {xls.sheet_names}')

# 일반현황 시트 로딩
df_ltc_fac = pd.read_excel(latest_file, sheet_name=xls.sheet_names[0])
print(f'일반현황: {df_ltc_fac.shape}')
print(f'컬럼: {list(df_ltc_fac.columns)}')
df_ltc_fac.head(3)

로딩: 국민건강보험공단_장기요양기관 시설별 현황_20250401.xlsx
시트: ['일반현황', '입소인원', '인력현황', '기관유형코드 정의']
일반현황: (31281, 10)
컬럼: ['장기요양기관코드', '장기요양기관이름', '우편번호', '시도코드', '시군구코드', '법정동코드', '시도 시군구 법정동명', '지정일자', '설치신고일자', '기관별 상세주소']


,장기요양기관코드,장기요양기관이름,우편번호,시도코드,시군구코드,법정동코드,시도 시군구 법정동명,지정일자,설치신고일자,기관별 상세주소
0,11111000006,청운노인요양원,3001.0,11.0,110.0,182.0,서울특별시 종로구 구기동,20080625,20080625,서울특별시 종로구 비봉길 76 (구기동)
1,11111000056,인자인케어센터,3006.0,11.0,110.0,183.0,서울특별시 종로구 평창동,20100701,20100701,서울특별시 종로구 평창17길 26 (평창동)
2,11111000060,평창동시니어센터,3006.0,11.0,110.0,183.0,서울특별시 종로구 평창동,20101125,20101125,서울특별시 종로구 평창15길 10 (평창동)


In [64]:
# ── 시군구코드로 아산시 필터 ──
# 아산시 시군구코드 확인 (코드 정의 시트 활용)
code_sheet = [s for s in xls.sheet_names if '시군구코드' in s]
if code_sheet:
    df_codes = pd.read_excel(latest_file, sheet_name=code_sheet[0])
    asan_code = df_codes[df_codes.iloc[:, -1].astype(str).str.contains('아산', na=False)]
    print('아산시 코드:')
    print(asan_code)
else:
    # 주소 컬럼에서 필터
    addr_col = [c for c in df_ltc_fac.columns if '주소' in str(c) or '시군구' in str(c) or '법정동' in str(c)]
    print(f'주소/시군구 관련 컬럼: {addr_col}')
    if addr_col:
        df_ltc_fac_asan = df_ltc_fac[df_ltc_fac[addr_col[0]].astype(str).str.contains('아산', na=False)]
        print(f'아산시 장기요양기관: {len(df_ltc_fac_asan)}개')

주소/시군구 관련 컬럼: ['시군구코드', '법정동코드', '시도 시군구 법정동명', '기관별 상세주소']
아산시 장기요양기관: 0개


---
## 7. 아산시 기초생활보장 연령별 수급자수

132개 월별 CSV(cp949) → 하나로 합치고 연월별 시계열 구성

In [65]:
# ── ZIP 내 CSV 일괄 로딩 ──
welfare_dir = DATA_DIR / '아산시 기초생활보장 연령별 수급자수(일반)(201401~202412)'

csv_files = sorted(welfare_dir.glob('*.csv'))
print(f'CSV 파일: {len(csv_files)}개')

dfs = []
for f in csv_files:
    try:
        _df = pd.read_csv(f, encoding='cp949')
        dfs.append(_df)
    except Exception as e:
        print(f'  ⚠ {f.name}: {e}')

df_welfare = pd.concat(dfs, ignore_index=True)
print(f'\n통합 완료: {df_welfare.shape}')
print(f'컬럼: {list(df_welfare.columns)}')
print(f'통계연월 범위: {df_welfare["통계연월"].min()} ~ {df_welfare["통계연월"].max()}')
df_welfare.head()

CSV 파일: 132개

통합 완료: (3057066, 5)
컬럼: ['통계연월', '통계시도명', '통계시군구명', '연령', '수급자수']
통계연월 범위: 201401 ~ 202412


,통계연월,통계시도명,통계시군구명,연령,수급자수
0,201401,서울특별시,종로구,0,1
1,201401,서울특별시,종로구,1,5
2,201401,서울특별시,종로구,2,6
3,201401,서울특별시,종로구,3,4
4,201401,서울특별시,종로구,4,9


In [66]:
# ── 아산시 필터 & 정제 ──
df_welfare = df_welfare[df_welfare['통계시군구명'].str.contains('아산', na=False)].copy()

# 연월 → datetime
df_welfare['연월'] = pd.to_datetime(df_welfare['통계연월'].astype(str), format='%Y%m')
df_welfare['연도'] = df_welfare['연월'].dt.year

# 수급자수 숫자 변환
df_welfare['수급자수'] = pd.to_numeric(df_welfare['수급자수'], errors='coerce')

print(f'아산시 수급자: {df_welfare.shape[0]:,}행 | {df_welfare["연월"].min()} ~ {df_welfare["연월"].max()}')
df_welfare.head()

아산시 수급자: 13,641행 | 2014-01-01 00:00:00 ~ 2024-12-01 00:00:00


,통계연월,통계시도명,통계시군구명,연령,수급자수,연월,연도
13852,201401,충청남도,아산시,0,12,2014-01-01,2014
13853,201401,충청남도,아산시,1,14,2014-01-01,2014
13854,201401,충청남도,아산시,2,24,2014-01-01,2014
13855,201401,충청남도,아산시,3,35,2014-01-01,2014
13856,201401,충청남도,아산시,4,42,2014-01-01,2014


---
## 8. 2024 노인복지시설 현황 (총괄표)

시군구별 노인복지시설 총괄표. 3행 헤더 구조.

In [67]:
# ── XLSX 로딩 ──
welfare_fac_dir = DATA_DIR / '2024 노인복지시설 현황'
xlsx_files = sorted(welfare_fac_dir.glob('*.xlsx'))
print(f'XLSX 파일:')
for f in xlsx_files:
    print(f'  {f.name}')
    xls = pd.ExcelFile(f)
    print(f'    시트: {xls.sheet_names}')

XLSX 파일:
  0._2024_목차_이용안내_총괄표(시.군.구)_12.16.xlsx
    시트: ['3. 가.노인주거복지시설 총괄표(시･군･구)', '나. 노인의료복지시설 총괄표(시･군･구)', '다. 노인여가복지시설 총괄표(시･군･구)', '라. 재가노인복지시설 총괄표(시･군･구)', '마. 노인일자리지원기관 총괄표(시･군･구)', '바. 학대피해노인 전용쉼터 총괄표(시･군･구)', '사. 치매전담형 장기요양기관 총괄표(시･군･구)']
  0._2024_목차_이용안내_총괄표_12.16.xlsx
    시트: ['1. 노인복지시설의 종류 및 현황', '2. 가.노인주거복지시설 총괄표', '나. 노인의료복지시설 총괄표', '다. 노인여가복지시설 총괄표', '라. 재가노인복지시설 총괄표', '마. 노인일자리지원기관 총괄표', '바. 학대피해노인 전용쉼터 총괄표', '사. 치매전담형 장기요양기관 총괄표', '아. 치매전문교육 수료자 현황(2021년도)']


In [68]:
# ── 시군구별 총괄표 로딩 (재가노인복지시설) ──
# 헤더 3행 → header=[0,1,2] or skiprows
target_file = [f for f in xlsx_files if '시.군.구' in f.name or '시군구' in f.name]
if target_file:
    xls = pd.ExcelFile(target_file[0])
    # 재가노인복지시설 시트 찾기
    jae_sheet = [s for s in xls.sheet_names if '재가' in s]
    if jae_sheet:
        df_jae = pd.read_excel(target_file[0], sheet_name=jae_sheet[0], header=[0,1,2])
        print(f'재가노인복지시설 총괄: {df_jae.shape}')
        df_jae.head(3)
    else:
        print(f'재가 시트 없음. 사용 가능 시트: {xls.sheet_names}')
else:
    print(f'시군구 총괄표 파일 없음. 파일 목록: {[f.name for f in xlsx_files]}')

재가노인복지시설 총괄: (248, 34)


---
## 9. 전국 병의원 및 약국 (시점별)

2023.12 ~ 2026.3 시점별 데이터. 아산시 필터링 후 의료기관 현황 파악.

In [69]:
# ── 시점별 폴더 확인 ──
med_dir = DATA_DIR / '전국 병의원 및 약국'

timepoints = sorted(med_dir.iterdir())
print(f'시점: {len(timepoints)}개')
for tp in timepoints:
    if tp.is_dir():
        files = list(tp.glob('*'))
        print(f'  📁 {tp.name}/ ({len(files)}개 파일)')
    else:
        print(f'  📄 {tp.name} ({tp.stat().st_size/1024:.0f}KB)')

시점: 14개
  📁 전국 병의원 및 약국 현황 2023.12/ (13개 파일)
  📄 전국 병의원 및 약국 현황 2023.12.zip (57613KB)
  📁 전국 병의원 및 약국 현황 2024.12/ (12개 파일)
  📄 전국 병의원 및 약국 현황 2024.12.zip (59078KB)
  📁 전국 병의원 및 약국 현황 2025.12/ (1개 파일)
  📄 전국 병의원 및 약국 현황 2025.12.zip (66789KB)
  📁 전국 병의원 및 약국 현황 2025.3/ (12개 파일)
  📄 전국 병의원 및 약국 현황 2025.3.zip (63343KB)
  📁 전국 병의원 및 약국 현황 2025.6/ (12개 파일)
  📄 전국 병의원 및 약국 현황 2025.6.zip (65815KB)
  📁 전국 병의원 및 약국 현황 2025.9/ (12개 파일)
  📄 전국 병의원 및 약국 현황 2025.9.zip (65813KB)
  📁 전국 병의원 및 약국 현황 2026.3/ (1개 파일)
  📄 전국 병의원 및 약국 현황 2026.3.zip (66766KB)


In [70]:
# ── 최신 시점 병원정보 로딩 & 아산시 필터 ──
# 가장 최신 시점 폴더
latest_tp = [d for d in sorted(med_dir.iterdir()) if d.is_dir()][-1]
hospital_file = list(latest_tp.rglob('*병원정보*'))

if hospital_file:
    df_hospital = pd.read_excel(hospital_file[0])
    print(f'전국 병원: {df_hospital.shape}')
    print(f'컬럼: {list(df_hospital.columns[:15])}')
    
    # 아산시 필터 (시군구코드명 또는 주소)
    df_hospital_asan = df_hospital[
        df_hospital['시군구코드명'].astype(str).str.contains('아산', na=False)
    ].copy()
    print(f'아산시 병원: {len(df_hospital_asan)}개')
    print(df_hospital_asan['종별코드명'].value_counts())
else:
    print(f'병원정보 파일 없음. 폴더 내 파일: {list(latest_tp.glob("*"))}')

전국 병원: (79562, 30)
컬럼: ['암호화요양기호', '요양기관명', '종별코드', '종별코드명', '시도코드', '시도코드명', '시군구코드', '시군구코드명', '읍면동', '우편번호', '주소', '전화번호', '병원홈페이지', '개설일자', '총의사수']
아산시 병원: 385개
종별코드명
의원       170
치과의원      98
한의원       64
보건진료소     16
보건지소      11
병원         9
요양병원       6
한방병원       4
치과병원       3
정신병원       2
보건소        1
종합병원       1
Name: count, dtype: int64


---
## 10. 지역사회건강조사

아산시 건강지표 데이터. 변수명 매핑(영문→한글)은 명세서의 '변수명 매핑' 시트 참조.

In [71]:
# ── 폴더 구조 확인 ──
health_dir = DATA_DIR / '지역사회건강조사'

print('지역사회건강조사 파일:')
for p in sorted(health_dir.rglob('*')):
    if p.is_file():
        rel = p.relative_to(health_dir)
        print(f'  {rel} ({p.stat().st_size/1024:.0f}KB)')

지역사회건강조사 파일:
  chs14_k.txt (7083KB)
  chs15_k.txt (6871KB)
  chs16_k.txt (5850KB)
  chs17_k.txt (7815KB)
  chs18_k.txt (8953KB)
  chs19_k.txt (8525KB)
  chs20_K_re.txt (5711KB)
  chs21_k.txt (6377KB)
  chs22_k.txt (5857KB)
  chs23_k.txt (6337KB)
  chs24_Chungnam.txt (7391KB)
  chs25_Chungnam.txt (7050KB)


In [72]:
# ── 변수명 매핑 로딩 (명세서) ──
df_varmap = pd.read_excel(
    DATA_DIR / '아산시_돌봄_데이터_명세서.xlsx',
    sheet_name='변수명 매핑 (영문-한글)'
)

health_vars = df_varmap[df_varmap['데이터셋'] == '지역사회건강조사']
print(f'지역사회건강조사 변수 매핑: {len(health_vars)}개')
print(health_vars[['영문 변수명', '한글 변수명', '분류']].to_string(index=False))

지역사회건강조사 변수 매핑: 209개
         영문 변수명          한글 변수명      분류
    EXAMIN_YEAR            조사연도    기본정보
      exmprs_no        조사대상자 번호    기본정보
            age              나이    기본정보
            sex              성별    기본정보
    CTPRVN_CODE            시도코드    기본정보
    PBHLTH_CODE           보건소코드    기본정보
        SPOT_NO          조사구 번호    기본정보
     HSHLD_CODE            가구코드    기본정보
     MBHLD_CODE           가구원코드    기본정보
   DONG_TY_CODE          동 유형코드    기본정보
  HOUSE_TY_CODE          주거유형코드    기본정보
    signgu_code           시군구코드    기본정보
        kstrata            층화변수  가중치/설계
           wt_h          가구 가중치  가중치/설계
           wt_p          개인 가중치  가중치/설계
       mbhld_co           가구원 수    기본정보
reside_adult_co        성인 동거인 수    기본정보
       fma_19z3          가구 월소득 가구/인구특성
       fma_04z1            교육수준 가구/인구특성
       fma_12z1            결혼상태 가구/인구특성
       fma_13z1         경제활동 여부 가구/인구특성
       fma_14z1           직업 분류 가구/인구특성
       fma_24z2        거주기간 (년) 가구/인구특성
       fma_27z1    

In [73]:
# ── 지역사회건강조사 로딩 (TSV, 연도별 컬럼 상이) ──
# 시군구코드: 2014~2018 → h_admincode=44250 / 2019~2025 → signgu_code=44270

health_files = sorted(health_dir.glob('chs*.txt'))
print(f'파일: {len(health_files)}개')

dfs_health = []
for f in health_files:
    # 인코딩 자동 감지
    for enc in ['utf-8', 'cp949']:
        try:
            df = pd.read_csv(f, sep='\t', encoding=enc, low_memory=False)
            break
        except (UnicodeDecodeError, Exception):
            continue

    # 시군구코드 컬럼 통일 (h_admincode → signgu_code)
    if 'h_admincode' in df.columns and 'signgu_code' not in df.columns:
        df = df.rename(columns={'h_admincode': 'signgu_code'})

    # 연도 컬럼 통일
    year_col = [c for c in df.columns if c.lower() in ('josa_year', 'examin_code', 'examin_year')]
    if year_col:
        df = df.rename(columns={year_col[0]: 'year'})

    # 아산시 필터 (44250=구코드, 44270=신코드)
    df_asan = df[df['signgu_code'].isin([44250, 44270])].copy()

    print(f'  {f.name}: enc={enc}, {df.shape[1]}열, 전체 {len(df):,}행 → 아산 {len(df_asan):,}행')
    dfs_health.append(df_asan)

# 공통 컬럼으로 병합
common_cols = set(dfs_health[0].columns)
for d in dfs_health[1:]:
    common_cols &= set(d.columns)
common_cols = sorted(common_cols)
print(f'\n공통 컬럼: {len(common_cols)}개')

df_health = pd.concat([d[common_cols] for d in dfs_health], ignore_index=True)
print(f'통합 완료: {df_health.shape[0]:,}행 × {df_health.shape[1]}열')
print(f'연도 분포:\n{df_health["year"].value_counts().sort_index()}')

파일: 12개
  chs14_k.txt: enc=utf-8, 216열, 전체 13,521행 → 아산 1,793행
  chs15_k.txt: enc=utf-8, 219열, 전체 13,587행 → 아산 1,793행
  chs16_k.txt: enc=utf-8, 182열, 전체 13,530행 → 아산 1,791행
  chs17_k.txt: enc=cp949, 251열, 전체 13,496행 → 아산 1,792행
  chs18_k.txt: enc=cp949, 286열, 전체 13,488행 → 아산 1,793행
  chs19_k.txt: enc=cp949, 280열, 전체 13,484행 → 아산 1,793행
  chs20_K_re.txt: enc=utf-8, 171열, 전체 13,434행 → 아산 1,784행
  chs21_k.txt: enc=utf-8, 193열, 전체 13,474행 → 아산 1,798행
  chs22_k.txt: enc=utf-8, 161열, 전체 14,341행 → 아산 1,791행
  chs23_k.txt: enc=utf-8, 178열, 전체 14,329행 → 아산 1,785행
  chs24_Chungnam.txt: enc=utf-8, 209열, 전체 14,335행 → 아산 1,789행
  chs25_Chungnam.txt: enc=utf-8, 201열, 전체 14,343행 → 아산 1,790행

공통 컬럼: 36개
통합 완료: 21,492행 × 36열
연도 분포:
year
2014    1793
2015    1793
2016    1791
2017    1792
2018    1793
2019    1793
2020    1784
2021    1798
2022    1791
2023    1785
2024    1789
2025    1790
Name: count, dtype: int64


In [74]:
# 변수명 한글 매핑 적용
rename_dict = dict(zip(
    health_vars['영문 변수명'].str.lower(),
    health_vars['한글 변수명']
))
df_health = df_health.rename(columns={c: rename_dict.get(c.lower(), c) for c in df_health.columns})

print(f'\n한글 매핑 적용 후 컬럼 예시: {list(df_health.columns[:15])}')


한글 매핑 적용 후 컬럼 예시: ['나이', '당뇨병 현재 치료', '당뇨 합병증 경험', '저혈당 경험', '평생 음주 경험', '1회 음주량', '교육수준', '고혈압 현재 치료', '인플루엔자 접종', '인플루엔자 접종 비용', '층화변수', '스트레스 인지', '우울증 진단', '우울증 현재 치료', '채소 섭취 빈도']


---
## 11. 파생 데이터 생성 & 병합

### 11-1. 읍면동별 인구 통합 시계열

In [75]:
# ── 인구총조사 + 주민등록 → 통합 시계열 ──
census_pop = df_census[df_census['항목'] == '총인구 (명)'][['읍면동', '연도', '값']].copy()
census_pop = census_pop.rename(columns={'값': '인구수'})
census_pop['출처'] = '인구총조사'

resident_total = df_resident_long[
    (df_resident_long['연령별'] == '합계') & (df_resident_long['항목'] == '총인구(명)')
][['읍면동', '연도', '값']].copy()
resident_total = resident_total.rename(columns={'값': '인구수'})
resident_total['출처'] = '주민등록'

pop_unified = pd.concat([census_pop, resident_total], ignore_index=True)
pop_unified = pop_unified.sort_values(['읍면동', '연도']).reset_index(drop=True)

print(f'읍면동별 인구 통합: {pop_unified.shape[0]}행 | 읍면동 {pop_unified["읍면동"].nunique()}개')
pop_unified.head(10)

읍면동별 인구 통합: 102행 | 읍면동 17개


,읍면동,연도,인구수,출처
0,도고면,2015,4756.0,인구총조사
1,도고면,2015,4756.0,주민등록
2,도고면,2020,4578.0,인구총조사
3,도고면,2020,4578.0,주민등록
4,도고면,2023,4428.0,주민등록
5,도고면,2024,4533.0,주민등록
6,둔포면,2015,13592.0,인구총조사
7,둔포면,2015,13592.0,주민등록
8,둔포면,2020,27006.0,인구총조사
9,둔포면,2020,27006.0,주민등록


### 11-2. 읍면동별 고령화율

In [76]:
# ── 65세이상 소계 직접 사용 (개별 연령대 합산 시 중복 방지) ──
elderly_pop = df_resident_long[
    (df_resident_long['연령별'] == '65세이상') & (df_resident_long['항목'] == '총인구(명)')
][['읍면동', '연도', '값']].copy()
elderly_pop = elderly_pop.rename(columns={'값': '고령인구'})

total_pop = df_resident_long[
    (df_resident_long['연령별'] == '합계') & (df_resident_long['항목'] == '총인구(명)')
][['읍면동', '연도', '값']].rename(columns={'값': '총인구'})

aging_ratio = elderly_pop.merge(total_pop, on=['읍면동', '연도'], how='inner')
aging_ratio['고령화율'] = (aging_ratio['고령인구'] / aging_ratio['총인구'] * 100).round(2)

print(f'고령화율: 평균 {aging_ratio["고령화율"].mean():.1f}%')
print(aging_ratio.sort_values('고령화율', ascending=False).head(10))

고령화율: 평균 19.6%
    읍면동    연도    고령인구     총인구   고령화율
60  도고면  2024  2018.0  4533.0  44.52
43  도고면  2023  1971.0  4428.0  44.51
59  선장면  2024  1375.0  3159.0  43.53
42  선장면  2023  1381.0  3181.0  43.41
26  도고면  2020  1904.0  4578.0  41.59
25  선장면  2020  1323.0  3408.0  38.82
53  송악면  2024  1464.0  3860.0  37.93
36  송악면  2023  1427.0  3950.0  36.13
57  영인면  2024  2157.0  6084.0  35.45
40  영인면  2023  2084.0  6025.0  34.59


### 11-3. 장기요양 아산시 등급 추이

In [77]:
# ── 아산시 + 급여유형 '계' 만 ──
ltc_asan = df_ltc[(df_ltc['시군구'] == '아산시') & (df_ltc['급여유형'] == '계')].copy()
ltc_summary = ltc_asan.groupby(['연도', '등급', '성별'])['인원'].sum().reset_index()

print(f'장기요양 아산시: {ltc_summary.shape[0]}행')
ltc_summary.head(10)

장기요양 아산시: 252행


,연도,등급,성별,인원
0,2014,1등급,남자,32
1,2014,1등급,여자,103
2,2014,1등급,합계,135
3,2014,2등급,남자,86
4,2014,2등급,여자,301
5,2014,2등급,합계,387
6,2014,3등급,남자,203
7,2014,3등급,여자,486
8,2014,3등급,합계,689
9,2014,4등급,남자,139


### 11-4. 충남 장래 고령화율 전망

In [78]:
# ── 65세 이상 합산 (80세이상 소계 제외하여 중복 방지) ──
proj_elderly_labels = [a for a in df_proj_long['연령대'].unique() if any(
    a.startswith(f'{x} ') or a.startswith(f'{x}~') for x in range(65, 105)
)]
proj_elderly_labels = [a for a in proj_elderly_labels if a != '80세이상']
print(f'고령 연령대 (중복 제거): {proj_elderly_labels}')

proj_elderly = df_proj_long[
    (df_proj_long['연령대'].isin(proj_elderly_labels)) & (df_proj_long['성별'] == '계')
].groupby('연도')['인구수'].sum().reset_index()
proj_elderly.columns = ['연도', '고령인구_추계']

proj_total = df_proj_long[
    (df_proj_long['연령대'] == '계') & (df_proj_long['성별'] == '계')
][['연도', '인구수']].rename(columns={'인구수': '총인구_추계'})

proj_aging = proj_elderly.merge(proj_total, on='연도')
proj_aging['고령화율_추계'] = (proj_aging['고령인구_추계'] / proj_aging['총인구_추계'] * 100).round(2)

print(f'\n충남 장래 고령화율:')
print(proj_aging[proj_aging['연도'].isin([2025, 2030, 2035, 2040, 2045, 2050])].to_string(index=False))

고령 연령대 (중복 제거): ['65 - 69세', '70 - 74세', '75 - 79세', '80 - 84세', '85 - 89세', '90 - 94세', '95 - 99세']

충남 장래 고령화율:
  연도  고령인구_추계  총인구_추계  고령화율_추계
2025   485982 2229051    21.80
2030   601332 2253655    26.68
2035   709263 2271887    31.22
2040   816694 2274956    35.90
2045   890111 2256841    39.44
2050   944539 2211343    42.71


### 11-5. 시설 접근성 (인구 대비)

In [79]:
# ── 읍면동별 시설수 vs 인구 ──
facility_total = df_facility.groupby('읍면동').size().reset_index(name='총시설수')

latest_pop = total_pop[total_pop['연도'] == total_pop['연도'].max()].copy()
access = facility_total.merge(latest_pop, on='읍면동', how='outer')
access['인구천명당_시설수'] = (access['총시설수'] / access['총인구'] * 1000).round(2)

print(f'시설접근성: {access.shape[0]}개 읍면동')
print(access.sort_values('인구천명당_시설수', ascending=False).to_string(index=False))

시설접근성: 19개 읍면동
 읍면동  총시설수   연도     총인구  인구천명당_시설수
 도고면   6.0 2024  4533.0       1.32
 염치읍   2.0 2024  5851.0       0.34
 인주면   2.0 2024  6753.0       0.30
 송악면   1.0 2024  3860.0       0.26
 둔포면   5.0 2024 32129.0       0.16
 영인면   1.0 2024  6084.0       0.16
 신창면   4.0 2024 35532.0       0.11
 배방읍  10.0 2024 92653.0       0.11
 음봉면   1.0 2024 25320.0       0.04
 탕정면   2.0 2024 49648.0       0.04
118동   2.0  NaN     NaN        NaN
 상가동   2.0  NaN     NaN        NaN
 선장면   NaN 2024  3159.0        NaN
온양1동   NaN 2024 10023.0        NaN
온양2동   NaN 2024 11852.0        NaN
온양3동   NaN 2024 38920.0        NaN
온양4동   NaN 2024 18263.0        NaN
온양5동   NaN 2024 21566.0        NaN
온양6동   NaN 2024 26363.0        NaN


---
## 12. 전처리 결과 저장

In [81]:
# ── 전처리 결과 저장 ──
outputs = {
    '01_인구총조사_long.csv': df_census,
    '02_주민등록인구_long.csv': df_resident_long,
    '03_장래인구추계_long.csv': df_proj_long,
    '04_장기요양_long.csv': df_ltc,
    '05_재가노인복지시설_아산.csv': df_facility,
    '06_읍면동별_인구통합.csv': pop_unified,
    '07_읍면동별_고령화율.csv': aging_ratio,
    '08_장기요양_아산_등급추이.csv': ltc_summary,
    '09_충남_장래_고령화율.csv': proj_aging,
    '10_시설접근성.csv': access,
    '11_지역사회건강조사_아산.csv': df_health,
    '12_기초생활_수급자_아산.csv': df_welfare,
}

for fname, df in outputs.items():
    path = OUT_DIR / fname
    df.to_csv(path, index=False, encoding='cp949')
    print(f'✓ {fname} ({df.shape[0]:,}행 × {df.shape[1]}열)')

print(f'\n저장 완료 → {OUT_DIR}')

✓ 01_인구총조사_long.csv (680행 × 4열)
✓ 02_주민등록인구_long.csv (14,382행 × 5열)
✓ 03_장래인구추계_long.csv (2,691행 × 5열)
✓ 04_장기요양_long.csv (1,253행 × 7열)
✓ 05_재가노인복지시설_아산.csv (84행 × 6열)
✓ 06_읍면동별_인구통합.csv (102행 × 4열)
✓ 07_읍면동별_고령화율.csv (68행 × 5열)
✓ 08_장기요양_아산_등급추이.csv (252행 × 4열)
✓ 09_충남_장래_고령화율.csv (39행 × 4열)
✓ 10_시설접근성.csv (19행 × 5열)
✓ 11_지역사회건강조사_아산.csv (21,492행 × 36열)
✓ 12_기초생활_수급자_아산.csv (13,641행 × 7열)

저장 완료 → C:\Users\HP\IdeaProjects\sundo\asan_care\asan_care\preprocessed
